# Decrypt the dataset

We need the key to decrypt the dataset, but it is stored in a remote attester (Trustee), and it will be provided to us only if **attestation** is successful, meaning the software & hardware of the CoCo pod and CVM have not been tampered with.

By doing this we ensure that the CoCo CVM is **safe** and having the right hardware and software running prevents any attacker from fetching the transactions while they are being read by the model (**data in use** security). This is possible because the hardware inside the CVM makes sure that all data being loaded in the memory is encrypted, so that if an attacker tries to do a physical/virtual memory dump, the output will only be encrypted/zeroed blobs of memory.

This is what **Confidential Computing** is about: securing data in use.

## Sealed secret

The podspec used to deploy this demo was already configured to load a sealed secret. This secret is populated by the CoCo CVM internal components by automatically performing attestation when booting the pod.

Let's now decrypt the dataset. Encryption key is retrieved automatically from Trustee!

In [ ]:
%%bash

KEY_FILE=/sealed/decryption/key
DATASET_SRC=downloaded_datasets
DATASET_DEST=datasets_dec

mkdir -p $DATASET_DEST
rm -rf $DATASET_DEST/*

for file in $DATASET_SRC/*; do
    fname=$(basename $file)
    fname=${fname%.enc}
    openssl enc -d -aes-256-cfb -pbkdf2 -kfile $KEY_FILE -in $file -out $DATASET_DEST/$fname
    echo "Decrypted" $DATASET_DEST/$fname
done

ls $DATASET_DEST

Checking back in our Trustee running in the secure environment, we will see that attestation happened and the key was successfully sent to this notebook:

```
[2025-04-24T13:02:44Z INFO  actix_web::middleware::logger] 10.88.0.14 "POST /kbs/v0/auth HTTP/1.1" 200 74 "-" "attestation-agent-kbs-client/0.1.0" 0.000246
[2025-04-24T13:02:48Z INFO  attestation_service] AzTdxVtpm Verifier/endorsement check passed.
[2025-04-24T13:02:48Z INFO  actix_web::middleware::logger] 10.88.0.14 "POST /kbs/v0/attest HTTP/1.1" 200 6384 "-" "attestation-agent-kbs-client/0.1.0" 1.211521
[2025-04-24T13:02:49Z INFO  actix_web::middleware::logger] 10.88.0.15 "GET /kbs/v0/resource/default/fraud-dataset-key/dataset_key HTTP/1.1" 200 530 "-" "attestation-agent-kbs-client/0.1.0" 0.001043
```

Now let's see if we can read the content of the dataset

In [ ]:
%%bash

DATASET_SRC=downloaded_datasets
DATASET_DEST=datasets_dec

for file in $DATASET_DEST/*; do
	head -n 5 $file
    echo ""
done

We have successfully managed to decrypt the dataset!